In [3]:
# ============================================================
# SMAP L4 — CHECK LIVE DATA AVAILABILITY
# ============================================================

import ee
from datetime import datetime, timedelta

ee.Initialize()

# Joshimath
JOSHIMATH_LON = 79.566
JOSHIMATH_LAT = 30.556

joshimath_point = ee.Geometry.Point(
    [JOSHIMATH_LON, JOSHIMATH_LAT]
)

study_area = joshimath_point.buffer(10000)

# Check last 7 days
end_date = datetime.now()
start_date = end_date - timedelta(days=7)

start_string = start_date.strftime("%Y-%m-%d")
end_string = end_date.strftime("%Y-%m-%d")

print("Checking SMAP L4:")
print(start_string, "to", end_string)

smap_l4 = (
    ee.ImageCollection(
        "NASA/SMAP/SPL4SMGP/008"
    )
    .filterDate(
        start_string,
        end_string
    )
    .filterBounds(
        study_area
    )
)

count = smap_l4.size().getInfo()

print()
print("SMAP L4 images:", count)


Checking SMAP L4:
2026-08-30 to 2026-09-06

SMAP L4 images: 32


In [4]:
# ============================================================
# SHOW SMAP L4 AVAILABLE DATES
# ============================================================

timestamps = (
    smap_l4
    .aggregate_array("system:time_start")
    .getInfo()
)

print("Number of observations:", len(timestamps))
print()

for timestamp in timestamps[-10:]:

    date = datetime.fromtimestamp(
        timestamp / 1000
    )

    print(
        date.strftime("%Y-%m-%d %H:%M")
    )

Number of observations: 32

2026-09-02 01:00
2026-09-02 04:00
2026-09-02 07:00
2026-09-02 10:00
2026-09-02 13:00
2026-09-02 16:00
2026-09-02 19:00
2026-09-02 22:00
2026-09-03 01:00
2026-09-03 04:00


In [5]:
# ============================================================
# GET LATEST SMAP L4 SOIL MOISTURE
# ============================================================

latest_smap = (
    smap_l4
    .sort(
        "system:time_start",
        False
    )
    .first()
)

# Surface soil moisture: 0–5 cm
soil_moisture = (
    latest_smap
    .select("sm_surface")
    .reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=study_area,
        scale=11000,
        maxPixels=1e7
    )
    .get("sm_surface")
)

soil_moisture_value = soil_moisture.getInfo()

print("Latest SMAP L4 surface soil moisture:")
print(soil_moisture_value)

Latest SMAP L4 surface soil moisture:
0.3242964607258742


In [6]:
# ============================================================
# GET SMAP L4 OBSERVATION TIME
# ============================================================

timestamp = (
    latest_smap
    .get("system:time_start")
    .getInfo()
)

observation_time = datetime.fromtimestamp(
    timestamp / 1000
)

print("SMAP L4 observation:")
print(
    observation_time.strftime(
        "%Y-%m-%d %H:%M:%S"
    )
)

print()
print("Soil moisture:", soil_moisture_value)

SMAP L4 observation:
2026-09-03 04:00:00

Soil moisture: 0.3242964607258742


In [7]:
# ============================================================
# FAST LIVE DATA — GPM + SMAP L4 + SENTINEL-1
# ============================================================

def get_latest_live_data():

    print("🔄 Searching for latest available satellite data...")

    # --------------------------------------------------------
    # Search period
    # --------------------------------------------------------

    end_date = ee.Date(
        datetime.now().strftime("%Y-%m-%d")
    )

    # Search previous 7 days
    start_date = end_date.advance(-7, "day")


    # ========================================================
    # 1. GPM — LATEST AVAILABLE
    # ========================================================

    gpm = (
        ee.ImageCollection(
            "NASA/GPM_L3/IMERG_V07"
        )
        .filterDate(start_date, end_date)
        .filterBounds(study_area)
        .sort(
            "system:time_start",
            False
        )
    )

    gpm_count = gpm.size()

    latest_gpm = gpm.first()

    rainfall = ee.Algorithms.If(
        gpm_count.gt(0),

        latest_gpm
        .select("precipitation")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=10000,
            maxPixels=1e7
        )
        .get("precipitation"),

        None
    )

    gpm_time = ee.Algorithms.If(
        gpm_count.gt(0),
        latest_gpm.get("system:time_start"),
        None
    )


    # ========================================================
    # 2. SMAP L4 — LATEST AVAILABLE
    # ========================================================

    smap = (
        ee.ImageCollection(
            "NASA/SMAP/SPL4SMGP/008"
        )
        .filterDate(start_date, end_date)
        .filterBounds(study_area)
        .sort(
            "system:time_start",
            False
        )
    )

    smap_count = smap.size()

    latest_smap = smap.first()

    soil_moisture = ee.Algorithms.If(
        smap_count.gt(0),

        latest_smap
        .select("sm_surface")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=11000,
            maxPixels=1e7
        )
        .get("sm_surface"),

        None
    )

    smap_time = ee.Algorithms.If(
        smap_count.gt(0),
        latest_smap.get("system:time_start"),
        None
    )


    # ========================================================
    # 3. SENTINEL-1 — LATEST AVAILABLE
    # ========================================================

    sentinel1 = (
        ee.ImageCollection(
            "COPERNICUS/S1_GRD"
        )
        .filterDate(start_date, end_date)
        .filterBounds(study_area)
        .filter(
            ee.Filter.eq(
                "instrumentMode",
                "IW"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VV"
            )
        )
        .filter(
            ee.Filter.listContains(
                "transmitterReceiverPolarisation",
                "VH"
            )
        )
        .sort(
            "system:time_start",
            False
        )
    )

    sentinel_count = sentinel1.size()

    latest_sentinel = sentinel1.first()

    sentinel_vv = ee.Algorithms.If(
        sentinel_count.gt(0),

        latest_sentinel
        .select("VV")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=30,
            maxPixels=1e7
        )
        .get("VV"),

        None
    )

    sentinel_vh = ee.Algorithms.If(
        sentinel_count.gt(0),

        latest_sentinel
        .select("VH")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=30,
            maxPixels=1e7
        )
        .get("VH"),

        None
    )

    sentinel_time = ee.Algorithms.If(
        sentinel_count.gt(0),
        latest_sentinel.get("system:time_start"),
        None
    )


    # ========================================================
    # 4. ELEVATION
    # ========================================================

    elevation_image = (
        ee.Image(
            "USGS/SRTMGL1_003"
        )
        .select("elevation")
    )

    elevation = (
        elevation_image
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=30,
            maxPixels=1e7
        )
        .get("elevation")
    )


    # ========================================================
    # 5. SLOPE
    # ========================================================

    slope_image = ee.Terrain.slope(
        elevation_image
    )

    slope = (
        slope_image
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=30,
            maxPixels=1e7
        )
        .get("slope")
    )


    # ========================================================
    # 6. ONE FINAL GEE REQUEST
    # ========================================================

    result = ee.Dictionary({

        "retrieval_date":
            datetime.now().strftime("%Y-%m-%d"),

        "rainfall":
            rainfall,

        "soil_moisture":
            soil_moisture,

        "sentinel1_vv":
            sentinel_vv,

        "sentinel1_vh":
            sentinel_vh,

        "elevation":
            elevation,

        "slope":
            slope,

        "gpm_time":
            gpm_time,

        "smap_time":
            smap_time,

        "sentinel1_time":
            sentinel_time,

        "gpm_count":
            gpm_count,

        "smap_count":
            smap_count,

        "sentinel1_count":
            sentinel_count

    }).getInfo()


    # ========================================================
    # RETURN
    # ========================================================

    print("✅ Latest live data retrieved")

    return result

In [8]:
# ============================================================
# TEST LATEST LIVE DATA
# ============================================================

live_result = get_latest_live_data()

print()
print("==============================================")
print("LATEST LIVE SATELLITE DATA")
print("==============================================")

for key, value in live_result.items():
    print(f"{key}: {value}")

🔄 Searching for latest available satellite data...
✅ Latest live data retrieved

LATEST LIVE SATELLITE DATA
elevation: 2731.5671510280154
gpm_count: 284
gpm_time: 1788561000000
rainfall: 0.5637121000911767
retrieval_date: 2026-09-06
sentinel1_count: 2
sentinel1_time: 1788395672000
sentinel1_vh: -19.077466735018895
sentinel1_vv: -12.266255153361978
slope: 35.31415903321138
smap_count: 32
smap_time: 1788388200000
soil_moisture: 0.3242964607258742


In [9]:
# ============================================================
# FLOOD MODEL — LOAD SAVED MODEL
# ============================================================

import joblib
import pandas as pd
import numpy as np

FLOOD_MODEL_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\Historical_Flood"
    r"\historical_flood_best_model.pkl"
)

flood_model = joblib.load(FLOOD_MODEL_PATH)

print("✅ Flood model loaded")
print("Model:", type(flood_model))

✅ Flood model loaded
Model: <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [10]:
# ============================================================
# CHECK FLOOD MODEL FEATURES
# ============================================================

print("Number of input features:",
      flood_model.n_features_in_)

if hasattr(flood_model, "feature_names_in_"):

    print("\nFeatures expected by flood model:")

    for i, feature in enumerate(
        flood_model.feature_names_in_,
        start=1
    ):
        print(f"{i}. {feature}")

else:

    print(
        "\n⚠️ Model does not contain feature names."
    )

Number of input features: 4

Features expected by flood model:
1. flooded_pixels
2. max_flood_duration_days
3. year
4. month


In [11]:
# ============================================================
# FLOOD MODEL — LIVE INPUT
# ============================================================

live_flood_df = pd.DataFrame([{
    "rainfall": live_result["rainfall"],
    "soil_moisture": live_result["soil_moisture"],
    "sentinel1_vv": live_result["sentinel1_vv"],
    "sentinel1_vh": live_result["sentinel1_vh"],
    "elevation": live_result["elevation"],
    "slope": live_result["slope"]
}])

print("✅ Live flood input created")
display(live_flood_df)

✅ Live flood input created


,rainfall,soil_moisture,sentinel1_vv,sentinel1_vh,elevation,slope
0,0.563712,0.324296,-12.266255,-19.077467,2731.567151,35.314159


In [12]:
# ============================================================
# FIND ALL SAVED MODELS IN YOUR PROJECT
# ============================================================

import os

PROJECT_FOLDER = r"E:\MY projects\Flash Flood Prediction Project"

model_extensions = (
    ".pkl",
    ".joblib",
    ".pt",
    ".pth",
    ".h5",
    ".keras"
)

print("SEARCHING FOR SAVED MODELS...")
print("=" * 70)

for root, dirs, files in os.walk(PROJECT_FOLDER):

    for file in files:

        if file.lower().endswith(model_extensions):

            full_path = os.path.join(root, file)

            print(full_path)

SEARCHING FOR SAVED MODELS...
E:\MY projects\Flash Flood Prediction Project\models\DEM\DEM_best_model.pkl
E:\MY projects\Flash Flood Prediction Project\models\DEM\DEM_gru.pt
E:\MY projects\Flash Flood Prediction Project\models\DEM\DEM_lstm.pt
E:\MY projects\Flash Flood Prediction Project\models\DEM\DEM_scaler.pkl
E:\MY projects\Flash Flood Prediction Project\models\Historical_Flood\historical_flood_best_model.pkl
E:\MY projects\Flash Flood Prediction Project\models\Historical_Flood\historical_flood_imputer.pkl
E:\MY projects\Flash Flood Prediction Project\models\Historical_Flood\historical_flood_logistic.pkl
E:\MY projects\Flash Flood Prediction Project\models\Historical_Flood\historical_flood_mlp.pkl
E:\MY projects\Flash Flood Prediction Project\models\Historical_Flood\historical_flood_random_forest.pkl
E:\MY projects\Flash Flood Prediction Project\models\Historical_Flood\historical_flood_scaler.pkl
E:\MY projects\Flash Flood Prediction Project\models\Historical_Landslide\landslide_be

In [13]:
# ============================================================
# LOAD BEST SAVED MODELS ONLY
# ============================================================

import os
import joblib
import torch

PROJECT_FOLDER = r"E:\MY projects\Flash Flood Prediction Project"

# ============================================================
# MODEL PATHS
# ============================================================

RAINFALL_MODEL_PATH = os.path.join(
    PROJECT_FOLDER,
    "models",
    "Rainfall",
    "Rainfall_Best_Model.pt"
)

SOIL_MODEL_PATH = os.path.join(
    PROJECT_FOLDER,
    "models",
    "Siol Moisture",
    "best_model_gru.pt"
)

SLOPE_MODEL_PATH = os.path.join(
    PROJECT_FOLDER,
    "models",
    "Slope",
    "Slope_best_model.pkl"
)

DEM_MODEL_PATH = os.path.join(
    PROJECT_FOLDER,
    "models",
    "DEM",
    "DEM_best_model.pkl"
)

FLOOD_MODEL_PATH = os.path.join(
    PROJECT_FOLDER,
    "models",
    "Historical_Flood",
    "historical_flood_best_model.pkl"
)

LANDSLIDE_MODEL_PATH = os.path.join(
    PROJECT_FOLDER,
    "models",
    "Historical_Landslide",
    "landslide_best_model.pt"
)

# ============================================================
# LOAD SKLEARN BEST MODELS
# ============================================================

slope_model = joblib.load(
    SLOPE_MODEL_PATH
)

dem_model = joblib.load(
    DEM_MODEL_PATH
)

flood_model = joblib.load(
    FLOOD_MODEL_PATH
)

print("✅ Slope best model loaded")
print("   ", type(slope_model))

print("✅ DEM best model loaded")
print("   ", type(dem_model))

print("✅ Historical flood best model loaded")
print("   ", type(flood_model))

✅ Slope best model loaded
    <class 'statsmodels.tsa.arima.model.ARIMAResultsWrapper'>
✅ DEM best model loaded
    <class 'statsmodels.tsa.arima.model.ARIMAResultsWrapper'>
✅ Historical flood best model loaded
    <class 'sklearn.ensemble._forest.RandomForestClassifier'>


In [14]:
# ============================================================
# INSPECT THE BEST PYTORCH MODELS
# ============================================================

import torch
import os

pt_models = {
    "Rainfall": RAINFALL_MODEL_PATH,
    "Soil Moisture": SOIL_MODEL_PATH,
    "Landslide": LANDSLIDE_MODEL_PATH
}

for name, path in pt_models.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("File:", path)
    print("Exists:", os.path.exists(path))

    if not os.path.exists(path):
        continue

    try:

        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        print("Loaded successfully")
        print("Python type:", type(checkpoint))

        if isinstance(checkpoint, dict):

            print("\nDictionary keys:")

            for key in checkpoint.keys():
                print("  ", key)

        else:

            print("\nObject:")
            print(checkpoint)

    except Exception as e:

        print("❌ Could not load")
        print("Error:", e)


Rainfall
File: E:\MY projects\Flash Flood Prediction Project\models\Rainfall\Rainfall_Best_Model.pt
Exists: True
Loaded successfully
Python type: <class 'dict'>

Dictionary keys:
   model_type
   model_state_dict
   input_size
   hidden_size
   num_layers
   sequence_length
   scaler

Soil Moisture
File: E:\MY projects\Flash Flood Prediction Project\models\Siol Moisture\best_model_gru.pt
Exists: True
Loaded successfully
Python type: <class 'dict'>

Dictionary keys:
   model_type
   input_size
   hidden_size
   num_layers
   sequence_length
   model_state_dict

Landslide
File: E:\MY projects\Flash Flood Prediction Project\models\Historical_Landslide\landslide_best_model.pt
Exists: True
Loaded successfully
Python type: <class 'dict'>

Dictionary keys:
   model_type
   input_size
   hidden_size
   num_layers
   sequence_length
   model_state_dict
   risk_score
   risk_level


In [15]:
# ============================================================
# INSPECT THE THREE BEST PYTORCH MODEL CHECKPOINTS
# ============================================================

import torch

checkpoints = {
    "Rainfall": RAINFALL_MODEL_PATH,
    "Soil Moisture": SOIL_MODEL_PATH,
    "Landslide": LANDSLIDE_MODEL_PATH
}

for name, path in checkpoints.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Model type       :", checkpoint.get("model_type"))
    print("Input size       :", checkpoint.get("input_size"))
    print("Hidden size      :", checkpoint.get("hidden_size"))
    print("Number of layers :", checkpoint.get("num_layers"))
    print("Sequence length  :", checkpoint.get("sequence_length"))

    if "scaler" in checkpoint:
        print("Scaler included  : YES")
    else:
        print("Scaler included  : NO")

    if name == "Landslide":
        print("Risk score       :", checkpoint.get("risk_score"))
        print("Risk level       :", checkpoint.get("risk_level"))


Rainfall
Model type       : LSTM
Input size       : 1
Hidden size      : 64
Number of layers : 2
Sequence length  : 24
Scaler included  : YES

Soil Moisture
Model type       : GRU
Input size       : 1
Hidden size      : 64
Number of layers : 2
Sequence length  : 30
Scaler included  : NO

Landslide
Model type       : GRU
Input size       : 1
Hidden size      : 32
Number of layers : 2
Sequence length  : 12
Scaler included  : NO
Risk score       : 0.0005532411742024124
Risk level       : LOW


In [16]:
# ============================================================
# INSPECT STATE DICT SHAPES
# ============================================================

for name, path in checkpoints.items():

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    state_dict = checkpoint["model_state_dict"]

    for key, value in state_dict.items():

        print(
            f"{key:40s} "
            f"{tuple(value.shape)}"
        )


Rainfall
lstm.weight_ih_l0                        (256, 1)
lstm.weight_hh_l0                        (256, 64)
lstm.bias_ih_l0                          (256,)
lstm.bias_hh_l0                          (256,)
lstm.weight_ih_l1                        (256, 64)
lstm.weight_hh_l1                        (256, 64)
lstm.bias_ih_l1                          (256,)
lstm.bias_hh_l1                          (256,)
fc.weight                                (1, 64)
fc.bias                                  (1,)

Soil Moisture
gru.weight_ih_l0                         (192, 1)
gru.weight_hh_l0                         (192, 64)
gru.bias_ih_l0                           (192,)
gru.bias_hh_l0                           (192,)
gru.weight_ih_l1                         (192, 64)
gru.weight_hh_l1                         (192, 64)
gru.bias_ih_l1                           (192,)
gru.bias_hh_l1                           (192,)
fc.weight                                (1, 64)
fc.bias                                  

In [17]:
# ============================================================
# RAINFALL — LOAD SAVED BEST LSTM MODEL
# ============================================================

import torch
import torch.nn as nn
import numpy as np

RAINFALL_MODEL_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\Rainfall\Rainfall_Best_Model.pt"
)

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# ------------------------------------------------------------
# LSTM architecture — EXACTLY AS TRAINED
# ------------------------------------------------------------

class RainfallLSTM(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=64,
        num_layers=2
    ):

        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        output, _ = self.lstm(x)

        last_output = output[:, -1, :]

        return self.fc(last_output)


# ------------------------------------------------------------
# Load checkpoint
# ------------------------------------------------------------

checkpoint = torch.load(
    RAINFALL_MODEL_PATH,
    map_location=device,
    weights_only=False
)

# ------------------------------------------------------------
# Create model
# ------------------------------------------------------------

rainfall_model = RainfallLSTM(
    input_size=checkpoint["input_size"],
    hidden_size=checkpoint["hidden_size"],
    num_layers=checkpoint["num_layers"]
).to(device)

# ------------------------------------------------------------
# Load trained weights
# ------------------------------------------------------------

rainfall_model.load_state_dict(
    checkpoint["model_state_dict"]
)

rainfall_model.eval()

print("✅ Rainfall LSTM loaded successfully")

print("Model type:",
      checkpoint["model_type"])

print("Sequence length:",
      checkpoint["sequence_length"])

print("Input size:",
      checkpoint["input_size"])

print("Hidden size:",
      checkpoint["hidden_size"])

print("Layers:",
      checkpoint["num_layers"])

Device: cuda
✅ Rainfall LSTM loaded successfully
Model type: LSTM
Sequence length: 24
Input size: 1
Hidden size: 64
Layers: 2


In [18]:
# ============================================================
# RAINFALL — LOAD TRAINING SCALER
# ============================================================

rainfall_scaler = checkpoint["scaler"]

print("✅ Rainfall scaler loaded")

print("Scaler mean:",
      rainfall_scaler.mean_)

print("Scaler scale:",
      rainfall_scaler.scale_)

✅ Rainfall scaler loaded
Scaler mean: [0.33901418]
Scaler scale: [1.72635692]


In [19]:
live_result["rainfall"]

0.5637121000911767

In [20]:
# ============================================================
# GET LAST 24 HOURLY GPM VALUES
# ============================================================

def get_last_24_rainfall():

    end_date = ee.Date(
        datetime.now().strftime("%Y-%m-%d")
    )

    start_date = end_date.advance(
        -3,
        "day"
    )

    gpm_collection = (
        ee.ImageCollection(
            "NASA/GPM_L3/IMERG_V07"
        )
        .filterDate(
            start_date,
            end_date
        )
        .filterBounds(
            study_area
        )
        .sort(
            "system:time_start"
        )
    )

    def extract_rainfall(image):

        rainfall = (
            image
            .select("precipitation")
            .reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=study_area,
                scale=10000,
                maxPixels=1e7
            )
            .get("precipitation")
        )

        return ee.Feature(
            None,
            {
                "rainfall": rainfall,
                "timestamp":
                    image.get("system:time_start")
            }
        )

    features = (
        gpm_collection
        .map(extract_rainfall)
        .filter(
            ee.Filter.notNull(
                ["rainfall"]
            )
        )
        .sort("timestamp")
    )

    data = features.getInfo()

    rainfall_values = []

    for feature in data["features"]:

        properties = feature["properties"]

        rainfall_values.append({
            "timestamp":
                properties["timestamp"],

            "rainfall":
                properties["rainfall"]
        })

    # Keep latest 24 observations
    rainfall_values = rainfall_values[-24:]

    return rainfall_values

In [21]:
# ============================================================
# TEST 24-HOUR RAINFALL DATA
# ============================================================

rainfall_history = get_last_24_rainfall()

print(
    "Number of rainfall observations:",
    len(rainfall_history)
)

for item in rainfall_history:

    observation_time = datetime.fromtimestamp(
        item["timestamp"] / 1000
    )

    print(
        observation_time.strftime(
            "%Y-%m-%d %H:%M"
        ),
        "→",
        item["rainfall"]
    )

Number of rainfall observations: 24
2026-09-04 16:30 → 0.03910173084264324
2026-09-04 17:00 → 0.007867965163880742
2026-09-04 17:30 → 0.00045454544438557195
2026-09-04 18:00 → 0.00037878785621036184
2026-09-04 18:30 → 0
2026-09-04 19:00 → 0
2026-09-04 19:30 → 0
2026-09-04 20:00 → 0.004285714189921108
2026-09-04 20:30 → 0.0794264040282601
2026-09-04 21:00 → 0.3724999998684054
2026-09-04 21:30 → 0.35869047274708227
2026-09-04 22:00 → 0.395075756717812
2026-09-04 22:30 → 1.2476731363635558
2026-09-04 23:00 → 0.8183549848340805
2026-09-04 23:30 → 0.2268614641257695
2026-09-05 00:00 → 0.23856059626325382
2026-09-05 00:30 → 1.0242532212045288
2026-09-05 01:00 → 0.6521644809674391
2026-09-05 01:30 → 0.6232142773541539
2026-09-05 02:00 → 0.6010822400902258
2026-09-05 02:30 → 0.5753138372553636
2026-09-05 03:00 → 0.5705086514353752
2026-09-05 03:30 → 1.000616851565126
2026-09-05 04:00 → 0.5637121000911767


In [22]:
# ============================================================
# RAINFALL LSTM — PREDICT NEXT VALUE
# ============================================================

if len(rainfall_history) < 24:

    print(
        "❌ Not enough rainfall observations."
    )

    print(
        "Required: 24"
    )

    print(
        "Available:",
        len(rainfall_history)
    )

else:

    # --------------------------------------------------------
    # Extract rainfall values
    # --------------------------------------------------------

    rainfall_values = np.array(
        [
            item["rainfall"]
            for item in rainfall_history
        ],
        dtype=np.float32
    ).reshape(-1, 1)

    # --------------------------------------------------------
    # Apply EXACT training scaler
    # --------------------------------------------------------

    rainfall_scaled = (
        rainfall_scaler
        .transform(
            rainfall_values
        )
        .flatten()
    )

    # --------------------------------------------------------
    # Create LSTM input
    # Shape = [batch, sequence, features]
    #        [1, 24, 1]
    # --------------------------------------------------------

    X_live = torch.tensor(
        rainfall_scaled,
        dtype=torch.float32
    ).reshape(
        1,
        24,
        1
    ).to(device)

    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    with torch.no_grad():

        predicted_scaled = (
            rainfall_model(
                X_live
            )
            .cpu()
            .numpy()
        )

    # --------------------------------------------------------
    # Convert prediction back to original rainfall scale
    # --------------------------------------------------------

    predicted_rainfall = (
        rainfall_scaler
        .inverse_transform(
            predicted_scaled
        )[0][0]
    )

    print("==========================================")
    print("🌧️ RAINFALL PREDICTION")
    print("==========================================")
    print(
        "Predicted next rainfall:",
        float(predicted_rainfall)
    )

🌧️ RAINFALL PREDICTION
Predicted next rainfall: 0.26999056339263916


In [23]:
# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================

%pip install -U earthengine-api pandas numpy scikit-learn joblib tensorflow

Note: you may need to restart the kernel to use updated packages.


In [25]:
# ============================================================
# SOIL MOISTURE — LIVE DATA + NEXT-DAY PREDICTION
# ============================================================

import os
import numpy as np
import pandas as pd
import ee
import tensorflow as tf
import joblib

from datetime import datetime, timedelta


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = r"E:\MY projects\Flash Flood Prediction Project"

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "best_model_gru.pt"
)

SCALER_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "soil_moisture_scaler.pkl"
)


# ============================================================
# 2. JOSHIMATH
# ============================================================

JOSHIMATH_LON = 79.566
JOSHIMATH_LAT = 30.556

joshimath_point = ee.Geometry.Point(
    [JOSHIMATH_LON, JOSHIMATH_LAT]
)

study_area = joshimath_point.buffer(10000)


# ============================================================
# 3. CHECK MODEL FILES
# ============================================================

print("=" * 60)
print("CHECKING SOIL MOISTURE MODEL")
print("=" * 60)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Soil moisture model not found:\n{MODEL_PATH}"
    )

if not os.path.exists(SCALER_PATH):
    raise FileNotFoundError(
        f"Soil moisture scaler not found:\n{SCALER_PATH}"
    )

print("✅ Model found:")
print(MODEL_PATH)

print()

print("✅ Scaler found:")
print(SCALER_PATH)


# ============================================================
# 4. LOAD MODEL
# ============================================================

soil_model = tf.keras.models.load_model(
    MODEL_PATH
)

print()
print("✅ Soil moisture GRU loaded")

print(
    "Model input shape:",
    soil_model.input_shape
)

print(
    "Model output shape:",
    soil_model.output_shape
)


# ============================================================
# 5. LOAD TRAINING SCALER
# ============================================================

soil_scaler = joblib.load(
    SCALER_PATH
)

print()
print("✅ Training StandardScaler loaded")

print(
    "Scaler mean:",
    soil_scaler.mean_
)

print(
    "Scaler scale:",
    soil_scaler.scale_
)


# ============================================================
# 6. GET SMAP L4 DATA
#
# We request 45 days instead of exactly 30 days.
# This gives us enough observations if some days
# have missing data.
# ============================================================

end_date = datetime.now()

start_date = (
    end_date - timedelta(days=45)
)

start_string = start_date.strftime(
    "%Y-%m-%d"
)

end_string = (
    end_date + timedelta(days=1)
).strftime(
    "%Y-%m-%d"
)

print()
print("=" * 60)
print("SEARCHING SMAP L4")
print("=" * 60)

print(
    "Period:",
    start_string,
    "to",
    end_string
)


# ============================================================
# 7. SMAP L4 COLLECTION
# ============================================================

smap_collection = (
    ee.ImageCollection(
        "NASA/SMAP/SPL4SMGP/008"
    )
    .filterDate(
        start_string,
        end_string
    )
    .filterBounds(
        study_area
    )
)


# ============================================================
# 8. COUNT OBSERVATIONS
# ============================================================

smap_count = (
    smap_collection
    .size()
    .getInfo()
)

print()
print(
    "SMAP L4 images:",
    smap_count
)


if smap_count == 0:

    raise RuntimeError(
        "❌ No SMAP L4 data available."
    )


# ============================================================
# 9. EXTRACT DAILY VALUES
#
# Instead of downloading every pixel/image,
# we calculate ONE mean value for Joshimath
# from each SMAP image.
# ============================================================

def extract_smap_value(image):

    value = (
        image
        .select("sm_surface")
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=study_area,
            scale=11000,
            maxPixels=1e7
        )
        .get("sm_surface")
    )

    return ee.Feature(
        None,
        {
            "datetime":
                image.get(
                    "system:time_start"
                ),

            "soil_moisture":
                value
        }
    )


# ============================================================
# 10. CREATE FEATURE COLLECTION
# ============================================================

features = (
    smap_collection
    .map(extract_smap_value)
)


# ============================================================
# 11. GET ONLY SMALL TABLE
# ============================================================

feature_list = (
    features
    .getInfo()
    .get("features", [])
)


print()
print(
    "Downloaded observations:",
    len(feature_list)
)


# ============================================================
# 12. CONVERT TO DATAFRAME
# ============================================================

records = []

for feature in feature_list:

    props = feature.get(
        "properties",
        {}
    )

    timestamp = props.get(
        "datetime"
    )

    value = props.get(
        "soil_moisture"
    )

    if timestamp is None:
        continue

    if value is None:
        continue

    observation_datetime = (
        datetime.fromtimestamp(
            timestamp / 1000
        )
    )

    records.append(
        {
            "datetime":
                observation_datetime,

            "soil_moisture":
                float(value)
        }
    )


smap_df = pd.DataFrame(
    records
)


# ============================================================
# 13. CHECK DATA
# ============================================================

if smap_df.empty:

    raise RuntimeError(
        "❌ SMAP data was found, but no valid "
        "soil-moisture values were extracted."
    )


smap_df = smap_df.sort_values(
    "datetime"
)

smap_df = smap_df.drop_duplicates(
    subset=["datetime"]
)


print()
print("=" * 60)
print("SMAP DATA")
print("=" * 60)

print(
    smap_df.tail(10)
)


# ============================================================
# 14. CONVERT TO DAILY SOIL MOISTURE
#
# SAME preprocessing as training:
#
# resample("D")
# mean()
# interpolate()
# ffill()
# bfill()
# ============================================================

daily_soil = (
    smap_df
    .set_index("datetime")
    ["soil_moisture"]
    .resample("D")
    .mean()
    .interpolate(method="time")
    .ffill()
    .bfill()
    .reset_index()
)


print()
print("=" * 60)
print("DAILY SOIL MOISTURE")
print("=" * 60)

print(
    daily_soil.tail(15)
)


# ============================================================
# 15. CHECK WHETHER 30 DAYS ARE AVAILABLE
# ============================================================

SEQUENCE_LENGTH = 30

if len(daily_soil) < SEQUENCE_LENGTH:

    raise RuntimeError(
        f"❌ Only {len(daily_soil)} daily values available.\n"
        f"Need at least {SEQUENCE_LENGTH}."
    )


# ============================================================
# 16. TAKE LAST 30 DAYS
# ============================================================

latest_30 = (
    daily_soil
    .tail(SEQUENCE_LENGTH)
    .copy()
)


print()
print("=" * 60)
print("LATEST 30 DAYS USED BY MODEL")
print("=" * 60)

print(
    latest_30
)


# ============================================================
# 17. GET RAW VALUES
# ============================================================

soil_values = (
    latest_30[
        "soil_moisture"
    ]
    .values
    .reshape(-1, 1)
)


print()
print(
    "Raw sequence shape:",
    soil_values.shape
)


# ============================================================
# 18. USE THE SAME TRAINING SCALER
# ============================================================

soil_scaled = soil_scaler.transform(
    soil_values
)


print(
    "Scaled sequence shape:",
    soil_scaled.shape
)


# ============================================================
# 19. CREATE MODEL INPUT
#
# Required:
# (samples, 30, 1)
# ============================================================

X_live = soil_scaled.reshape(
    1,
    SEQUENCE_LENGTH,
    1
)


print(
    "Model input shape:",
    X_live.shape
)


# ============================================================
# 20. PREDICT NEXT DAY
# ============================================================

predicted_scaled = (
    soil_model
    .predict(
        X_live,
        verbose=0
    )
)


# ============================================================
# 21. CONVERT BACK TO ORIGINAL UNITS
# ============================================================

predicted_soil_moisture = (
    soil_scaler
    .inverse_transform(
        predicted_scaled
    )
)


predicted_value = float(
    predicted_soil_moisture[0][0]
)


# ============================================================
# 22. FINAL RESULT
# ============================================================

latest_date = (
    latest_30["datetime"]
    .iloc[-1]
)


prediction_date = (
    latest_date +
    timedelta(days=1)
)


print()
print("=" * 60)
print("🌱 SOIL MOISTURE PREDICTION")
print("=" * 60)

print(
    "Latest available date :",
    latest_date.strftime("%Y-%m-%d")
)

print(
    "Prediction date        :",
    prediction_date.strftime("%Y-%m-%d")
)

print(
    "Predicted soil moisture:",
    predicted_value
)

print()
print("✅ Soil moisture prediction completed")

CHECKING SOIL MOISTURE MODEL


FileNotFoundError: Soil moisture model not found:
E:\MY projects\Flash Flood Prediction Project\models\best_model_gru.pt

In [26]:
# ============================================================
# SOIL MOISTURE MODEL PATH
# ============================================================

MODEL_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\Siol Moisture\best_model_gru.pt"
)

SCALER_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\soil_moisture_scaler.pkl"
)

print("Model path:")
print(MODEL_PATH)

print("\nModel exists:", os.path.exists(MODEL_PATH))
print("Scaler exists:", os.path.exists(SCALER_PATH))

Model path:
E:\MY projects\Flash Flood Prediction Project\models\Siol Moisture\best_model_gru.pt

Model exists: True
Scaler exists: False


In [28]:
import torch

MODEL_PATH = r"E:\MY projects\Flash Flood Prediction Project\models\Siol Moisture\best_model_gru.pt"

checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=False
)

print("=" * 60)
print("SOIL MOISTURE MODEL CHECKPOINT")
print("=" * 60)

print("\nCheckpoint type:")
print(type(checkpoint))

print("\nCheckpoint keys:")
if isinstance(checkpoint, dict):
    for key in checkpoint.keys():
        print("  ", key)

print("\nModel information:")

if isinstance(checkpoint, dict):

    for key in [
        "model_type",
        "input_size",
        "hidden_size",
        "num_layers",
        "sequence_length"
    ]:
        print(
            f"{key}:",
            checkpoint.get(key, "NOT FOUND")
        )

print("\nState dictionary shapes:")

state_dict = checkpoint["model_state_dict"]

for key, value in state_dict.items():
    print(
        f"{key:40s}",
        tuple(value.shape)
    )

SOIL MOISTURE MODEL CHECKPOINT

Checkpoint type:
<class 'dict'>

Checkpoint keys:
   model_type
   input_size
   hidden_size
   num_layers
   sequence_length
   model_state_dict

Model information:
model_type: GRU
input_size: 1
hidden_size: 64
num_layers: 2
sequence_length: 30

State dictionary shapes:
gru.weight_ih_l0                         (192, 1)
gru.weight_hh_l0                         (192, 64)
gru.bias_ih_l0                           (192,)
gru.bias_hh_l0                           (192,)
gru.weight_ih_l1                         (192, 64)
gru.weight_hh_l1                         (192, 64)
gru.bias_ih_l1                           (192,)
gru.bias_hh_l1                           (192,)
fc.weight                                (1, 64)
fc.bias                                  (1,)


In [29]:
# ============================================================
# STEP — LOAD SOIL MOISTURE GRU MODEL
# ============================================================

import torch
import torch.nn as nn
import numpy as np

# ------------------------------------------------------------
# MODEL PATH
# ------------------------------------------------------------

SOIL_MODEL_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\Siol Moisture\best_model_gru.pt"
)

# ------------------------------------------------------------
# DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# ------------------------------------------------------------
# DEFINE EXACT SAME MODEL ARCHITECTURE
# ------------------------------------------------------------

class SoilMoistureGRU(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=64,
        num_layers=2
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )

    def forward(self, x):

        output, _ = self.gru(x)

        last_output = output[:, -1, :]

        return self.fc(last_output)


# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    SOIL_MODEL_PATH,
    map_location=device,
    weights_only=False
)

# ------------------------------------------------------------
# CREATE MODEL
# ------------------------------------------------------------

soil_model = SoilMoistureGRU(
    input_size=checkpoint["input_size"],
    hidden_size=checkpoint["hidden_size"],
    num_layers=checkpoint["num_layers"]
).to(device)

# ------------------------------------------------------------
# LOAD WEIGHTS
# ------------------------------------------------------------

soil_model.load_state_dict(
    checkpoint["model_state_dict"]
)

soil_model.eval()

# ------------------------------------------------------------
# INFORMATION
# ------------------------------------------------------------

print()
print("=" * 60)
print("SOIL MOISTURE MODEL LOADED")
print("=" * 60)

print("Model type       :", checkpoint["model_type"])
print("Input size       :", checkpoint["input_size"])
print("Hidden size      :", checkpoint["hidden_size"])
print("Number of layers :", checkpoint["num_layers"])
print("Sequence length  :", checkpoint["sequence_length"])
print("Device           :", device)

print()
print("✅ Soil moisture GRU loaded successfully")

Device: cuda

SOIL MOISTURE MODEL LOADED
Model type       : GRU
Input size       : 1
Hidden size      : 64
Number of layers : 2
Sequence length  : 30
Device           : cuda

✅ Soil moisture GRU loaded successfully


In [33]:
# ============================================================
# SOIL MOISTURE — CHECK CORRECT TRAINED MODEL
# ============================================================

import os
import joblib
import tensorflow as tf

MODEL_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\Soil_Moisture\soil_moisture_gru.keras"
)

SCALER_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\Soil_Moisture\soil_moisture_scaler.pkl"
)

print("=" * 60)
print("CHECKING SOIL MOISTURE MODEL")
print("=" * 60)

print("\nModel:")
print(MODEL_PATH)
print("Exists:", os.path.exists(MODEL_PATH))

print("\nScaler:")
print(SCALER_PATH)
print("Exists:", os.path.exists(SCALER_PATH))


if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        "soil_moisture_gru.keras not found"
    )

if not os.path.exists(SCALER_PATH):
    raise FileNotFoundError(
        "soil_moisture_scaler.pkl not found"
    )


# Load model
soil_model = tf.keras.models.load_model(
    MODEL_PATH
)

# Load scaler
soil_scaler = joblib.load(
    SCALER_PATH
)


print("\n✅ Soil moisture Keras model loaded")
print("Model type:", type(soil_model))

print("\nModel summary:")
soil_model.summary()

print("\n✅ Soil moisture scaler loaded")
print("Scaler type:", type(soil_scaler))

CHECKING SOIL MOISTURE MODEL

Model:
E:\MY projects\Flash Flood Prediction Project\models\Soil_Moisture\soil_moisture_gru.keras
Exists: True

Scaler:
E:\MY projects\Flash Flood Prediction Project\models\Soil_Moisture\soil_moisture_scaler.pkl
Exists: True

✅ Soil moisture Keras model loaded
Model type: <class 'keras.src.models.sequential.Sequential'>

Model summary:


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 30, 64)         │        12,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 32)             │         9,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 68,453 (267.40 KB)

 Trainable params: 22,817 (89.13 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 45,636 (178.27 KB)


✅ Soil moisture scaler loaded
Scaler type: <class 'sklearn.preprocessing._data.StandardScaler'>


In [34]:
# ============================================================
# SOIL MOISTURE — LIVE PREDICTION
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

# ============================================================
# 1. PATHS
# ============================================================

MODEL_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\Soil_Moisture\soil_moisture_gru.keras"
)

SCALER_PATH = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\models\Soil_Moisture\soil_moisture_scaler.pkl"
)

LIVE_CSV = (
    r"E:\MY projects\Flash Flood Prediction Project"
    r"\data\live_daily.csv"
)

SEQUENCE_LENGTH = 30


# ============================================================
# 2. LOAD MODEL
# ============================================================

print("=" * 60)
print("LOADING SOIL MOISTURE MODEL")
print("=" * 60)

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Model not found:\n{MODEL_PATH}"
    )

soil_model = tf.keras.models.load_model(
    MODEL_PATH
)

print("✅ Soil moisture GRU model loaded")


# ============================================================
# 3. LOAD SCALER
# ============================================================

if not os.path.exists(SCALER_PATH):
    raise FileNotFoundError(
        f"Scaler not found:\n{SCALER_PATH}"
    )

soil_scaler = joblib.load(
    SCALER_PATH
)

print("✅ Soil moisture scaler loaded")


# ============================================================
# 4. LOAD LIVE DATA
# ============================================================

if not os.path.exists(LIVE_CSV):
    raise FileNotFoundError(
        f"Live CSV not found:\n{LIVE_CSV}"
    )

live_df = pd.read_csv(
    LIVE_CSV
)

print()
print("Live dataset shape:", live_df.shape)
print("Columns:")
print(list(live_df.columns))


# ============================================================
# 5. CHECK SOIL MOISTURE COLUMN
# ============================================================

if "soil_moisture" not in live_df.columns:

    raise ValueError(
        "soil_moisture column not found in live_daily.csv"
    )


# ============================================================
# 6. CLEAN SOIL MOISTURE DATA
# ============================================================

soil_values = pd.to_numeric(
    live_df["soil_moisture"],
    errors="coerce"
)

soil_values = soil_values.dropna()


print()
print("Valid soil moisture records:", len(soil_values))


# ============================================================
# 7. CHECK 30 DAYS
# ============================================================

if len(soil_values) < SEQUENCE_LENGTH:

    raise ValueError(
        f"Need at least {SEQUENCE_LENGTH} valid "
        f"soil moisture values for prediction. "
        f"Currently available: {len(soil_values)}"
    )


# ============================================================
# 8. TAKE LAST 30 DAYS
# ============================================================

last_30_values = soil_values.values[
    -SEQUENCE_LENGTH:
]

print()
print("Last 30 soil moisture values:")
print(last_30_values)


# ============================================================
# 9. SCALE USING TRAINING SCALER
# ============================================================

last_30_scaled = soil_scaler.transform(
    last_30_values.reshape(-1, 1)
)


# ============================================================
# 10. CREATE MODEL INPUT
# ============================================================

X_live = last_30_scaled.reshape(
    1,
    SEQUENCE_LENGTH,
    1
)


print()
print("Model input shape:", X_live.shape)


# ============================================================
# 11. PREDICT NEXT SOIL MOISTURE
# ============================================================

predicted_scaled = soil_model.predict(
    X_live,
    verbose=0
)


# ============================================================
# 12. CONVERT BACK TO ORIGINAL SCALE
# ============================================================

predicted_soil_moisture = soil_scaler.inverse_transform(
    predicted_scaled.reshape(-1, 1)
)[0][0]


# ============================================================
# 13. FINAL RESULT
# ============================================================

print()
print("=" * 60)
print("🌱 SOIL MOISTURE LIVE PREDICTION")
print("=" * 60)

print(
    "Latest actual soil moisture:",
    float(last_30_values[-1])
)

print(
    "Predicted next soil moisture:",
    float(predicted_soil_moisture)
)

print("=" * 60)

LOADING SOIL MOISTURE MODEL
✅ Soil moisture GRU model loaded
✅ Soil moisture scaler loaded


FileNotFoundError: Live CSV not found:
E:\MY projects\Flash Flood Prediction Project\data\live_daily.csv

In [35]:
# ============================================================
# SAVE LATEST LIVE GEE DATA TO live_daily.csv
# ============================================================

import os
import pandas as pd

# ------------------------------------------------------------
# PATH
# ------------------------------------------------------------

PROJECT_DIR = r"E:\MY projects\Flash Flood Prediction Project"

DATA_DIR = os.path.join(
    PROJECT_DIR,
    "data"
)

LIVE_CSV = os.path.join(
    DATA_DIR,
    "live_daily.csv"
)

os.makedirs(
    DATA_DIR,
    exist_ok=True
)


# ============================================================
# GET LATEST LIVE DATA FROM GEE
# ============================================================

print("🔄 Getting latest live GEE data...")

live_result = get_latest_live_data()


# ============================================================
# DISPLAY LIVE DATA
# ============================================================

print("\n==============================================")
print("LATEST LIVE DATA")
print("==============================================")

for key, value in live_result.items():
    print(f"{key}: {value}")


# ============================================================
# CONVERT TO DATAFRAME
# ============================================================

new_df = pd.DataFrame(
    [live_result]
)


# ============================================================
# CHECK EXISTING CSV
# ============================================================

if os.path.exists(LIVE_CSV):

    print("\n📂 Existing live CSV found")

    old_df = pd.read_csv(
        LIVE_CSV
    )

    # Add new record
    combined_df = pd.concat(
        [
            old_df,
            new_df
        ],
        ignore_index=True
    )

else:

    print("\n📂 No live CSV found")
    print("🆕 Creating new live dataset...")

    combined_df = new_df


# ============================================================
# REMOVE DUPLICATES
# ============================================================

if "retrieval_date" in combined_df.columns:

    combined_df = combined_df.drop_duplicates(
        subset=["retrieval_date"],
        keep="last"
    )


# ============================================================
# SORT BY DATE
# ============================================================

if "retrieval_date" in combined_df.columns:

    combined_df = combined_df.sort_values(
        "retrieval_date"
    )


# ============================================================
# SAVE
# ============================================================

combined_df.to_csv(
    LIVE_CSV,
    index=False
)


# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n==============================================")
print("✅ LIVE DATA SAVED")
print("==============================================")

print("File:")
print(LIVE_CSV)

print("\nRecords:")
print(len(combined_df))

print("\nLatest record:")
display(
    combined_df.tail(1)
)

🔄 Getting latest live GEE data...
🔄 Searching for latest available satellite data...
✅ Latest live data retrieved

LATEST LIVE DATA
elevation: 2731.5671510280154
gpm_count: 284
gpm_time: 1788561000000
rainfall: 0.5637121000911767
retrieval_date: 2026-09-06
sentinel1_count: 2
sentinel1_time: 1788395672000
sentinel1_vh: -19.077466735018895
sentinel1_vv: -12.266255153361978
slope: 35.31415903321138
smap_count: 32
smap_time: 1788388200000
soil_moisture: 0.3242964607258742

📂 No live CSV found
🆕 Creating new live dataset...

✅ LIVE DATA SAVED
File:
E:\MY projects\Flash Flood Prediction Project\data\live_daily.csv

Records:
1

Latest record:


,elevation,gpm_count,gpm_time,rainfall,retrieval_date,sentinel1_count,sentinel1_time,sentinel1_vh,sentinel1_vv,slope,smap_count,smap_time,soil_moisture
0,2731.567151,284,1788561000000,0.563712,2026-09-06,2,1788395672000,-19.077467,-12.266255,35.314159,32,1788388200000,0.324296


In [37]:
# ============================================================
# SOIL MOISTURE — HISTORICAL + LIVE DATA FOR PREDICTION
# ============================================================

import os
import glob
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = r"E:\MY projects\Flash Flood Prediction Project"

HISTORICAL_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "soil_moisture_monthly"
)

LIVE_CSV = os.path.join(
    PROJECT_DIR,
    "data",
    "live_daily.csv"
)

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "Soil_Moisture",
    "soil_moisture_gru.keras"
)

SCALER_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "Soil_Moisture",
    "soil_moisture_scaler.pkl"
)

SEQUENCE_LENGTH = 30


# ============================================================
# 2. CHECK FILES
# ============================================================

print("=" * 60)
print("CHECKING SOIL MOISTURE FILES")
print("=" * 60)

print("Model exists :", os.path.exists(MODEL_PATH))
print("Scaler exists:", os.path.exists(SCALER_PATH))
print("Live CSV exists:", os.path.exists(LIVE_CSV))


# ============================================================
# 3. LOAD HISTORICAL SOIL MOISTURE
# ============================================================

print("\n" + "=" * 60)
print("LOADING HISTORICAL SOIL MOISTURE")
print("=" * 60)

csv_files = glob.glob(
    os.path.join(
        HISTORICAL_DIR,
        "**",
        "*.csv"
    ),
    recursive=True
)

print("Historical CSV files:", len(csv_files))

historical_frames = []

for file in csv_files:

    try:

        temp = pd.read_csv(file)

        if (
            "datetime" in temp.columns
            and
            "soil_moisture" in temp.columns
        ):

            temp = temp[
                [
                    "datetime",
                    "soil_moisture"
                ]
            ]

            historical_frames.append(temp)

    except Exception as e:

        print("Error:", file)
        print(e)


if len(historical_frames) == 0:

    raise ValueError(
        "No historical soil moisture CSV files found."
    )


historical_df = pd.concat(
    historical_frames,
    ignore_index=True
)


# ============================================================
# 4. CLEAN HISTORICAL DATA
# ============================================================

historical_df["datetime"] = pd.to_datetime(
    historical_df["datetime"],
    errors="coerce"
)

historical_df["soil_moisture"] = pd.to_numeric(
    historical_df["soil_moisture"],
    errors="coerce"
)

historical_df = historical_df.dropna(
    subset=[
        "datetime",
        "soil_moisture"
    ]
)

historical_df = historical_df.sort_values(
    "datetime"
)


# ============================================================
# 5. CONVERT TO DAILY DATA
# ============================================================

historical_daily = (

    historical_df
    .set_index("datetime")
    ["soil_moisture"]
    .resample("D")
    .mean()

    .interpolate(
        method="time"
    )

    .ffill()
    .bfill()

    .reset_index()
)


print(
    "Historical daily records:",
    len(historical_daily)
)


# ============================================================
# 6. LOAD LIVE DATA
# ============================================================

print("\n" + "=" * 60)
print("LOADING LIVE DATA")
print("=" * 60)

if not os.path.exists(LIVE_CSV):

    raise FileNotFoundError(
        f"Live CSV not found:\n{LIVE_CSV}"
    )


live_df = pd.read_csv(
    LIVE_CSV
)


print(
    "Live records:",
    len(live_df)
)


# ============================================================
# 7. CONVERT LIVE DATE
# ============================================================

if "retrieval_date" in live_df.columns:

    live_df["datetime"] = pd.to_datetime(
        live_df["retrieval_date"],
        errors="coerce"
    )

elif "date" in live_df.columns:

    live_df["datetime"] = pd.to_datetime(
        live_df["date"],
        errors="coerce"
    )

else:

    raise ValueError(
        "Live CSV does not contain retrieval_date or date."
    )


live_df["soil_moisture"] = pd.to_numeric(
    live_df["soil_moisture"],
    errors="coerce"
)

live_soil = live_df[
    [
        "datetime",
        "soil_moisture"
    ]
].dropna()


# ============================================================
# 8. COMBINE HISTORICAL + LIVE
# ============================================================

combined_df = pd.concat(
    [
        historical_daily,
        live_soil
    ],
    ignore_index=True
)


# ============================================================
# 9. REMOVE DUPLICATE DATES
# ============================================================

combined_df = combined_df.sort_values(
    "datetime"
)

combined_df = combined_df.drop_duplicates(
    subset=["datetime"],
    keep="last"
)


# ============================================================
# 10. TAKE LAST 30 VALUES
# ============================================================

combined_df = combined_df.dropna(
    subset=["soil_moisture"]
)

if len(combined_df) < SEQUENCE_LENGTH:

    raise ValueError(
        f"Still not enough soil moisture data. "
        f"Available: {len(combined_df)}, "
        f"Required: {SEQUENCE_LENGTH}"
    )


last_30 = combined_df.tail(
    SEQUENCE_LENGTH
).copy()


print("\n" + "=" * 60)
print("LAST 30 SOIL MOISTURE VALUES")
print("=" * 60)

print(
    last_30.to_string(index=False)
)


# ============================================================
# 11. LOAD MODEL
# ============================================================

print("\n" + "=" * 60)
print("LOADING GRU MODEL")
print("=" * 60)

model = tf.keras.models.load_model(
    MODEL_PATH
)

print("✅ GRU model loaded")


# ============================================================
# 12. LOAD SCALER
# ============================================================

scaler = joblib.load(
    SCALER_PATH
)

print("✅ StandardScaler loaded")


# ============================================================
# 13. PREPARE 30-DAY INPUT
# ============================================================

soil_values = last_30[
    "soil_moisture"
].values.reshape(-1, 1)


soil_scaled = scaler.transform(
    soil_values
)


X_live = soil_scaled.reshape(
    1,
    SEQUENCE_LENGTH,
    1
)


print("\nInput shape:")
print(X_live.shape)


# ============================================================
# 14. PREDICT NEXT DAY SOIL MOISTURE
# ============================================================

predicted_scaled = model.predict(
    X_live,
    verbose=0
)


# ============================================================
# 15. CONVERT BACK TO ORIGINAL SCALE
# ============================================================

predicted_soil_moisture = scaler.inverse_transform(
    predicted_scaled
)[0][0]


# ============================================================
# 16. FINAL RESULT
# ============================================================

print("\n" + "=" * 60)
print("🌱 SOIL MOISTURE PREDICTION")
print("=" * 60)

print(
    "Latest actual soil moisture:",
    last_30["soil_moisture"].iloc[-1]
)

print(
    "Predicted next soil moisture:",
    float(predicted_soil_moisture)
)

print(
    "Prediction date:",
    (
        last_30["datetime"].iloc[-1]
        + pd.Timedelta(days=1)
    ).strftime("%Y-%m-%d")
)

print("=" * 60)
print("✅ SOIL MOISTURE PREDICTION COMPLETED")
print("=" * 60)

CHECKING SOIL MOISTURE FILES
Model exists : True
Scaler exists: True
Live CSV exists: True

LOADING HISTORICAL SOIL MOISTURE
Historical CSV files: 80
Error: E:\MY projects\Flash Flood Prediction Project\data\soil_moisture_monthly\soil_moisture_monthly\soil_moisture_2020_12.csv
No columns to parse from file
Error: E:\MY projects\Flash Flood Prediction Project\data\soil_moisture_monthly\soil_moisture_monthly\soil_moisture_2021_01.csv
No columns to parse from file
Error: E:\MY projects\Flash Flood Prediction Project\data\soil_moisture_monthly\soil_moisture_monthly\soil_moisture_2021_02.csv
No columns to parse from file
Error: E:\MY projects\Flash Flood Prediction Project\data\soil_moisture_monthly\soil_moisture_monthly\soil_moisture_2021_03.csv
No columns to parse from file
Error: E:\MY projects\Flash Flood Prediction Project\data\soil_moisture_monthly\soil_moisture_monthly\soil_moisture_2022_01.csv
No columns to parse from file
Error: E:\MY projects\Flash Flood Prediction Project\data\so

In [38]:
# ============================================================
# STEP 1 — CHECK REQUIRED PACKAGES
# ============================================================

import sys
import subprocess

packages = [
    "torch",
    "numpy",
    "pandas"
]

for package in packages:
    try:
        __import__(package)
        print(f"✅ {package} installed")
    except ImportError:
        print(f"Installing {package}...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

print("\n✅ Required packages are ready")

✅ torch installed
✅ numpy installed
✅ pandas installed

✅ Required packages are ready


In [39]:
# ============================================================
# STEP 2 — LOAD SAVED LANDSLIDE GRU MODEL
# ============================================================

import os
import torch
import torch.nn as nn

PROJECT_DIR = r"E:\MY projects\Flash Flood Prediction Project"

LANDSLIDE_MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "Historical_Landslide",
    "landslide_best_model.pt"
)

print("=" * 60)
print("LOADING LANDSLIDE MODEL")
print("=" * 60)

print("Model path:")
print(LANDSLIDE_MODEL_PATH)

print("\nModel exists:")
print(os.path.exists(LANDSLIDE_MODEL_PATH))


if not os.path.exists(LANDSLIDE_MODEL_PATH):

    raise FileNotFoundError(
        f"Landslide model not found:\n{LANDSLIDE_MODEL_PATH}"
    )


# ------------------------------------------------------------
# LOAD CHECKPOINT
# ------------------------------------------------------------

checkpoint = torch.load(
    LANDSLIDE_MODEL_PATH,
    map_location="cpu",
    weights_only=False
)


print("\nCheckpoint type:")
print(type(checkpoint))

print("\nCheckpoint keys:")

for key in checkpoint.keys():
    print("  ", key)


# ============================================================
# READ MODEL PARAMETERS
# ============================================================

INPUT_SIZE = checkpoint["input_size"]
HIDDEN_SIZE = checkpoint["hidden_size"]
NUM_LAYERS = checkpoint["num_layers"]
SEQUENCE_LENGTH = checkpoint["sequence_length"]

print("\n======================================")
print("MODEL INFORMATION")
print("======================================")

print("Model type       :", checkpoint["model_type"])
print("Input size       :", INPUT_SIZE)
print("Hidden size      :", HIDDEN_SIZE)
print("Number of layers :", NUM_LAYERS)
print("Sequence length  :", SEQUENCE_LENGTH)


# ============================================================
# DEFINE SAME GRU ARCHITECTURE
# ============================================================

class LandslideGRU(nn.Module):

    def __init__(
        self,
        input_size=1,
        hidden_size=32,
        num_layers=2
    ):

        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(
            hidden_size,
            1
        )


    def forward(self, x):

        output, _ = self.gru(x)

        last_output = output[:, -1, :]

        return self.fc(last_output)


# ============================================================
# CREATE MODEL
# ============================================================

landslide_model = LandslideGRU(
    input_size=INPUT_SIZE,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS
)


# ============================================================
# LOAD WEIGHTS
# ============================================================

landslide_model.load_state_dict(
    checkpoint["model_state_dict"]
)

landslide_model.eval()


print("\n✅ Landslide GRU model loaded successfully")
print(landslide_model)

LOADING LANDSLIDE MODEL
Model path:
E:\MY projects\Flash Flood Prediction Project\models\Historical_Landslide\landslide_best_model.pt

Model exists:
True

Checkpoint type:
<class 'dict'>

Checkpoint keys:
   model_type
   input_size
   hidden_size
   num_layers
   sequence_length
   model_state_dict
   risk_score
   risk_level

MODEL INFORMATION
Model type       : GRU
Input size       : 1
Hidden size      : 32
Number of layers : 2
Sequence length  : 12

✅ Landslide GRU model loaded successfully
LandslideGRU(
  (gru): GRU(1, 32, num_layers=2, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


In [40]:
# ============================================================
# STEP 3 — LOAD HISTORICAL LANDSLIDE DATA
# ============================================================

import glob
import zipfile
import pandas as pd
import numpy as np

ZIP_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "historical_landslide_monthly.zip"
)

EXTRACT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "historical_landslide_monthly"
)

os.makedirs(
    EXTRACT_DIR,
    exist_ok=True
)


# ============================================================
# EXTRACT ZIP IF NECESSARY
# ============================================================

if os.path.exists(ZIP_PATH):

    print("📦 Extracting historical landslide dataset...")

    with zipfile.ZipFile(
        ZIP_PATH,
        "r"
    ) as zip_ref:

        zip_ref.extractall(
            EXTRACT_DIR
        )

    print("✅ Dataset extracted")


# ============================================================
# FIND CSV FILES
# ============================================================

csv_files = glob.glob(
    os.path.join(
        EXTRACT_DIR,
        "**",
        "*.csv"
    ),
    recursive=True
)

print("\nCSV files found:", len(csv_files))


if len(csv_files) == 0:

    raise FileNotFoundError(
        "No historical landslide CSV files found."
    )


# ============================================================
# READ CSV FILES
# ============================================================

frames = []

for file in csv_files:

    try:

        temp = pd.read_csv(file)

        if "event_date" in temp.columns:

            frames.append(
                temp
            )

    except Exception as e:

        print("Error:", file)
        print(e)


if len(frames) == 0:

    raise ValueError(
        "No CSV containing event_date was found."
    )


df = pd.concat(
    frames,
    ignore_index=True
)


print("\nCombined dataset shape:")
print(df.shape)

print("\nColumns:")
print(df.columns.tolist())

📦 Extracting historical landslide dataset...
✅ Dataset extracted

CSV files found: 10

Combined dataset shape:
(12, 13)

Columns:
['event_date', 'latitude', 'longitude', 'landslide_type', 'trigger', 'source', 'location', 'country', 'admin_division', 'fatalities', 'injuries', 'year', 'month']


In [41]:
# ============================================================
# STEP 4 — CREATE MONTHLY LANDSLIDE OCCURRENCE
# ============================================================

# ------------------------------------------------------------
# DATE PROCESSING
# ------------------------------------------------------------

df["event_date"] = pd.to_datetime(
    df["event_date"],
    errors="coerce"
)

df = df.dropna(
    subset=["event_date"]
)

df = df.sort_values(
    "event_date"
)


# ============================================================
# YEAR / MONTH
# ============================================================

df["year"] = df[
    "event_date"
].dt.year

df["month"] = df[
    "event_date"
].dt.month


# ============================================================
# MONTHLY EVENT COUNT
# ============================================================

monthly_events = (

    df.groupby(
        [
            "year",
            "month"
        ]
    )

    .size()

    .reset_index(
        name="landslide_count"
    )
)


# ============================================================
# OCCURRENCE
# ============================================================

monthly_events[
    "landslide_occurrence"
] = (

    monthly_events[
        "landslide_count"
    ] > 0

).astype(int)


# ============================================================
# CREATE DATE COLUMN
# ============================================================

monthly_events["date"] = pd.to_datetime(
    monthly_events[
        ["year", "month"]
    ].assign(
        day=1
    )
)


monthly_events = monthly_events.sort_values(
    "date"
)


print("=" * 60)
print("MONTHLY LANDSLIDE DATA")
print("=" * 60)

print(
    monthly_events.tail(20).to_string(
        index=False
    )
)

print(
    "\nTotal monthly records:",
    len(monthly_events)
)

MONTHLY LANDSLIDE DATA
 year  month  landslide_count  landslide_occurrence       date
 2007      7                2                     1 2007-07-01
 2015      4                4                     1 2015-04-01

Total monthly records: 2


In [43]:
# ============================================================
# FIX LANDSLIDE DATA — CREATE COMPLETE MONTHLY TIMELINE
# ============================================================

import pandas as pd
import numpy as np


# ============================================================
# 1. MAKE SURE EVENT DATE IS DATETIME
# ============================================================

df["event_date"] = pd.to_datetime(
    df["event_date"],
    errors="coerce"
)

df = df.dropna(
    subset=["event_date"]
)


# ============================================================
# 2. GET MONTH OF EACH EVENT
# ============================================================

df["month_date"] = (
    df["event_date"]
    .dt.to_period("M")
    .dt.to_timestamp()
)


# ============================================================
# 3. COUNT LANDSLIDES PER MONTH
# ============================================================

monthly_counts = (
    df.groupby("month_date")
      .size()
      .reset_index(
          name="landslide_count"
      )
)


# ============================================================
# 4. CREATE COMPLETE MONTHLY DATE RANGE
# ============================================================

start_month = (
    df["event_date"]
    .min()
    .to_period("M")
    .to_timestamp()
)

end_month = (
    df["event_date"]
    .max()
    .to_period("M")
    .to_timestamp()
)


all_months = pd.date_range(
    start=start_month,
    end=end_month,
    freq="MS"
)


# ============================================================
# 5. REINDEX
#
# Months with no landslide = 0
# ============================================================

monthly_events = (
    monthly_counts
    .set_index("month_date")
    .reindex(all_months)
    .fillna(0)
    .rename_axis("date")
    .reset_index()
)


# ============================================================
# 6. INTEGER LANDSLIDE COUNT
# ============================================================

monthly_events["landslide_count"] = (
    monthly_events["landslide_count"]
    .astype(int)
)


# ============================================================
# 7. LANDSLIDE OCCURRENCE
#
# 0 = No landslide
# 1 = Landslide
# ============================================================

monthly_events["landslide_occurrence"] = (
    monthly_events["landslide_count"] > 0
).astype(int)


# ============================================================
# 8. YEAR / MONTH
# ============================================================

monthly_events["year"] = (
    monthly_events["date"].dt.year
)

monthly_events["month"] = (
    monthly_events["date"].dt.month
)


# ============================================================
# 9. DISPLAY DATA
# ============================================================

print("=" * 70)
print("COMPLETE MONTHLY LANDSLIDE DATA")
print("=" * 70)

print(
    monthly_events.tail(20).to_string(
        index=False
    )
)


print("\nTotal monthly records:")
print(len(monthly_events))


print("\nMonths with landslides:")
print(
    monthly_events[
        monthly_events["landslide_occurrence"] == 1
    ].shape[0]
)


print("\nMonths without landslides:")
print(
    monthly_events[
        monthly_events["landslide_occurrence"] == 0
    ].shape[0]
)


# ============================================================
# 10. CHECK WHETHER 12 MONTHS ARE AVAILABLE
# ============================================================

if len(monthly_events) >= 12:

    print("\n✅ At least 12 monthly observations available")

else:

    print(
        "\n❌ Still fewer than 12 months available"
    )

    print(
        "Available months:",
        len(monthly_events)
    )


# ============================================================
# 11. SHOW LAST 12 MONTHS
# ============================================================

if len(monthly_events) >= 12:

    last_12 = monthly_events.tail(
        12
    )

    print("\n" + "=" * 70)
    print("LAST 12 MONTHS")
    print("=" * 70)

    print(
        last_12[
            [
                "date",
                "landslide_count",
                "landslide_occurrence"
            ]
        ].to_string(
            index=False
        )
    )

COMPLETE MONTHLY LANDSLIDE DATA
      date  landslide_count  landslide_occurrence  year  month
2013-09-01                0                     0  2013      9
2013-10-01                0                     0  2013     10
2013-11-01                0                     0  2013     11
2013-12-01                0                     0  2013     12
2014-01-01                0                     0  2014      1
2014-02-01                0                     0  2014      2
2014-03-01                0                     0  2014      3
2014-04-01                0                     0  2014      4
2014-05-01                0                     0  2014      5
2014-06-01                0                     0  2014      6
2014-07-01                0                     0  2014      7
2014-08-01                0                     0  2014      8
2014-09-01                0                     0  2014      9
2014-10-01                0                     0  2014     10
2014-11-01             

In [44]:
# ============================================================
# STEP 5 — CREATE LAST 12-MONTH SEQUENCE
# ============================================================

landslide_values = monthly_events[
    "landslide_occurrence"
].values


if len(landslide_values) < SEQUENCE_LENGTH:

    raise ValueError(
        f"Need at least {SEQUENCE_LENGTH} "
        f"monthly landslide observations. "
        f"Available: {len(landslide_values)}"
    )


# ------------------------------------------------------------
# TAKE LAST 12 MONTHS
# ------------------------------------------------------------

last_12 = monthly_events.tail(
    SEQUENCE_LENGTH
).copy()


print("=" * 60)
print("LAST 12 MONTHS USED FOR PREDICTION")
print("=" * 60)

print(
    last_12[
        [
            "date",
            "landslide_count",
            "landslide_occurrence"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# CREATE NUMPY ARRAY
# ============================================================

X_live = last_12[
    "landslide_occurrence"
].values.astype(
    np.float32
)


# ============================================================
# RESHAPE
#
# (12,) → (1, 12, 1)
# ============================================================

X_live = X_live.reshape(
    1,
    SEQUENCE_LENGTH,
    1
)


print("\nInput shape:")
print(X_live.shape)

LAST 12 MONTHS USED FOR PREDICTION
      date  landslide_count  landslide_occurrence
2014-05-01                0                     0
2014-06-01                0                     0
2014-07-01                0                     0
2014-08-01                0                     0
2014-09-01                0                     0
2014-10-01                0                     0
2014-11-01                0                     0
2014-12-01                0                     0
2015-01-01                0                     0
2015-02-01                0                     0
2015-03-01                0                     0
2015-04-01                4                     1

Input shape:
(1, 12, 1)


In [45]:
# ============================================================
# LANDSLIDE GRU — LIVE PREDICTION
# ============================================================

import torch
import numpy as np


# ============================================================
# 1. CHECK DATA
# ============================================================

if len(monthly_events) < 12:

    raise ValueError(
        f"Need at least 12 months of historical "
        f"landslide data. Available: {len(monthly_events)}"
    )


# ============================================================
# 2. TAKE LAST 12 MONTHS
# ============================================================

last_12 = monthly_events.tail(
    12
).copy()


print("=" * 70)
print("12-MONTH INPUT TO LANDSLIDE GRU")
print("=" * 70)

print(
    last_12[
        [
            "date",
            "landslide_count",
            "landslide_occurrence"
        ]
    ].to_string(
        index=False
    )
)


# ============================================================
# 3. CREATE INPUT
# ============================================================

values = last_12[
    "landslide_occurrence"
].values.astype(
    np.float32
)


# Shape:
# (12,) → (1, 12, 1)

X_live = values.reshape(
    1,
    12,
    1
)


# ============================================================
# 4. CONVERT TO PYTORCH
# ============================================================

X_tensor = torch.tensor(
    X_live,
    dtype=torch.float32
)


# ============================================================
# 5. PREDICT
# ============================================================

landslide_model.eval()

with torch.no_grad():

    output = landslide_model(
        X_tensor
    )


# ============================================================
# 6. SIGMOID
# ============================================================

probability = torch.sigmoid(
    output
).item()


# ============================================================
# 7. RISK SCORE
# ============================================================

risk_score = float(
    np.clip(
        probability,
        0,
        1
    )
)


# ============================================================
# 8. RISK LEVEL
# ============================================================

if risk_score < 0.20:

    risk_level = "LOW"

elif risk_score < 0.40:

    risk_level = "MODERATE"

elif risk_score < 0.60:

    risk_level = "HIGH"

elif risk_score < 0.80:

    risk_level = "VERY HIGH"

else:

    risk_level = "EXTREME"


# ============================================================
# 9. FINAL RESULT
# ============================================================

print("\n")
print("=" * 70)
print("🌋 LANDSLIDE PREDICTION")
print("=" * 70)

print(
    "Probability :",
    probability
)

print(
    "Risk Score  :",
    risk_score
)

print(
    "Risk Level  :",
    risk_level
)

print(
    "Prediction for:",
    (
        last_12["date"].iloc[-1]
        + pd.DateOffset(months=1)
    ).strftime("%Y-%m")
)

print("=" * 70)

12-MONTH INPUT TO LANDSLIDE GRU
      date  landslide_count  landslide_occurrence
2014-05-01                0                     0
2014-06-01                0                     0
2014-07-01                0                     0
2014-08-01                0                     0
2014-09-01                0                     0
2014-10-01                0                     0
2014-11-01                0                     0
2014-12-01                0                     0
2015-01-01                0                     0
2015-02-01                0                     0
2015-03-01                0                     0
2015-04-01                4                     1


🌋 LANDSLIDE PREDICTION
Probability : 0.0005527944304049015
Risk Score  : 0.0005527944304049015
Risk Level  : LOW
Prediction for: 2015-05


In [46]:
# ============================================================
# STEP 1 — COMBINED LIVE RISK PREDICTION
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. CURRENT LIVE VALUES
# ============================================================

# These values should come from your previous prediction cells

rainfall_prediction = float(predicted_rainfall)

soil_moisture_prediction = float(predicted_soil_moisture)

landslide_probability = float(probability)


# ============================================================
# 2. LIVE GEE VALUES
# ============================================================

# Use the live_result dictionary created from GEE

sentinel1_vv = live_result["sentinel1_vv"]

sentinel1_vh = live_result["sentinel1_vh"]

slope = live_result["slope"]

elevation = live_result["elevation"]


# ============================================================
# 3. DISPLAY INPUTS
# ============================================================

print("=" * 60)
print("LIVE ENVIRONMENTAL INPUTS")
print("=" * 60)

print(
    "Predicted rainfall       :",
    rainfall_prediction
)

print(
    "Predicted soil moisture  :",
    soil_moisture_prediction
)

print(
    "Historical landslide prob:",
    landslide_probability
)

print(
    "Sentinel-1 VV            :",
    sentinel1_vv
)

print(
    "Sentinel-1 VH            :",
    sentinel1_vh
)

print(
    "Slope                    :",
    slope
)

print(
    "Elevation                :",
    elevation
)


# ============================================================
# 4. NORMALIZATION FUNCTIONS
# ============================================================

def normalize_rainfall(value):
    
    # Approximate rainfall risk scale
    # 0 mm -> 0 risk
    # 50+ mm -> 1 risk
    
    return np.clip(
        value / 50.0,
        0,
        1
    )


def normalize_soil_moisture(value):
    
    # Typical SMAP surface soil moisture
    # Approximate risk increases with wetness
    
    return np.clip(
        value / 0.5,
        0,
        1
    )


def normalize_slope(value):
    
    # 0 degree -> 0
    # 60+ degree -> 1
    
    return np.clip(
        value / 60.0,
        0,
        1
    )


def normalize_sentinel_vh(value):
    
    # More negative VH generally indicates lower backscatter.
    # Convert approximate range -30 to 0 dB into 0-1.
    
    return np.clip(
        (value + 30.0) / 30.0,
        0,
        1
    )


def normalize_sentinel_vv(value):
    
    return np.clip(
        (value + 30.0) / 30.0,
        0,
        1
    )


# ============================================================
# 5. CALCULATE INDIVIDUAL RISK SCORES
# ============================================================

rainfall_risk = normalize_rainfall(
    rainfall_prediction
)

soil_risk = normalize_soil_moisture(
    soil_moisture_prediction
)

slope_risk = normalize_slope(
    slope
)

vv_risk = normalize_sentinel_vv(
    sentinel1_vv
)

vh_risk = normalize_sentinel_vh(
    sentinel1_vh
)


# Historical landslide model already outputs 0-1

historical_landslide_risk = np.clip(
    landslide_probability,
    0,
    1
)


# ============================================================
# 6. FLOOD RISK SCORE
# ============================================================

flood_risk = (

    0.45 * rainfall_risk +

    0.25 * soil_risk +

    0.15 * slope_risk +

    0.15 * historical_landslide_risk

)


flood_risk = float(
    np.clip(
        flood_risk,
        0,
        1
    )
)


# ============================================================
# 7. LANDSLIDE RISK SCORE
# ============================================================

landslide_risk = (

    0.30 * soil_risk +

    0.20 * rainfall_risk +

    0.20 * slope_risk +

    0.10 * vv_risk +

    0.10 * vh_risk +

    0.10 * historical_landslide_risk

)


landslide_risk = float(
    np.clip(
        landslide_risk,
        0,
        1
    )
)


# ============================================================
# 8. OVERALL RISK
# ============================================================

overall_risk = (

    0.60 * flood_risk +

    0.40 * landslide_risk

)


overall_risk = float(
    np.clip(
        overall_risk,
        0,
        1
    )
)


# ============================================================
# 9. RISK LEVEL FUNCTION
# ============================================================

def get_risk_level(score):

    if score < 0.20:

        return "LOW"

    elif score < 0.40:

        return "MODERATE"

    elif score < 0.60:

        return "HIGH"

    elif score < 0.80:

        return "VERY HIGH"

    else:

        return "EXTREME"


flood_level = get_risk_level(
    flood_risk
)

landslide_level = get_risk_level(
    landslide_risk
)

overall_level = get_risk_level(
    overall_risk
)


# ============================================================
# 10. FINAL RESULT
# ============================================================

print()
print("=" * 60)
print("🚨 COMBINED RISK PREDICTION")
print("=" * 60)

print(
    "\nRainfall risk       :",
    round(rainfall_risk, 4)
)

print(
    "Soil moisture risk  :",
    round(soil_risk, 4)
)

print(
    "Slope risk          :",
    round(slope_risk, 4)
)

print(
    "Sentinel-1 VV risk  :",
    round(vv_risk, 4)
)

print(
    "Sentinel-1 VH risk  :",
    round(vh_risk, 4)
)

print(
    "Historical landslide:",
    round(historical_landslide_risk, 4)
)


print("\n----------------------------------------")

print(
    "🌊 Flood Risk        :",
    round(flood_risk, 4)
)

print(
    "Flood Risk Level     :",
    flood_level
)


print(
    "\n🌋 Landslide Risk     :",
    round(landslide_risk, 4)
)

print(
    "Landslide Risk Level :",
    landslide_level
)


print(
    "\n🚨 OVERALL RISK       :",
    round(overall_risk, 4)
)

print(
    "Overall Risk Level   :",
    overall_level
)

print("=" * 60)

LIVE ENVIRONMENTAL INPUTS
Predicted rainfall       : 0.26999056339263916
Predicted soil moisture  : 0.3014800548553467
Historical landslide prob: 0.0005527944304049015
Sentinel-1 VV            : -12.266255153361978
Sentinel-1 VH            : -19.077466735018895
Slope                    : 35.31415903321138
Elevation                : 2731.5671510280154

🚨 COMBINED RISK PREDICTION

Rainfall risk       : 0.0054
Soil moisture risk  : 0.603
Slope risk          : 0.5886
Sentinel-1 VV risk  : 0.5911
Sentinel-1 VH risk  : 0.3641
Historical landslide: 0.0006

----------------------------------------
🌊 Flood Risk        : 0.2415
Flood Risk Level     : MODERATE

🌋 Landslide Risk     : 0.3953
Landslide Risk Level : MODERATE

🚨 OVERALL RISK       : 0.303
Overall Risk Level   : MODERATE
